# Origination Feature Engineering and Leakage Controls

## Purpose

This notebook creates a loan-level modeling dataset by combining
validated origination characteristics with the validated 24-month
credit outcome.

## Modeling Design

- 2015 and 2016 will form the model-development population.
- 2017 will serve as the later-vintage test population.
- 2006 will be retained for historical stress comparison.
- The primary target is `default_24m`.
- Censored loans will remain documented but will not be used as known
  nondefault observations in supervised model training.

## Leakage-Control Principle

Only information known at or before loan origination may be used as a
model predictor.

Monthly performance fields, delinquency information, modifications,
zero-balance events, actual losses, servicing outcomes, and future
observation-window information are prohibited as predictors.

Loan identifiers, dates, and audit fields may be retained for
traceability but will not be supplied to the model.

In [2]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq


project_root = Path.cwd()

if project_root.name == "notebooks":
    project_root = project_root.parent

processed_data_directory = (
    project_root / "data" / "processed"
)

origination_path = (
    processed_data_directory
    / "originations_clean.parquet"
)

outcomes_path = (
    processed_data_directory
    / "loan_outcomes.parquet"
)

print("Project root:", project_root)
print(
    "Origination file exists:",
    origination_path.exists(),
)
print(
    "Outcome file exists:",
    outcomes_path.exists(),
)

Project root: c:\GitHub Projects\freddie-mac-credit-risk
Origination file exists: True
Outcome file exists: True


In [3]:
origination_columns = (
    pq.read_schema(origination_path).names
)

outcome_columns = (
    pq.read_schema(outcomes_path).names
)

print(
    "Origination columns:",
    len(origination_columns),
)

print(
    "Outcome columns:",
    len(outcome_columns),
)

display(
    pd.Series(
        origination_columns,
        name="origination_column",
    ).to_frame()
)

Origination columns: 33
Outcome columns: 18


,origination_column
0,vintage
1,credit_score
2,first_payment_date
3,first_time_homebuyer_indicator
4,maturity_date
5,msa
6,mi_percentage
7,number_of_units
8,occupancy_status
9,original_cltv


In [4]:
# Fields retained from origination data.
origination_fields = [
    # Traceability and population assignment
    "loan_identifier",
    "vintage",

    # Numeric origination characteristics
    "credit_score",
    "mi_percentage",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers",

    # Categorical origination characteristics
    "first_time_homebuyer_indicator",
    "number_of_units",
    "occupancy_status",
    "channel",
    "prepayment_penalty_indicator",
    "amortization_type",
    "property_state",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",

    # Audit field retained but not used as a predictor
    "extreme_ratio_review_flag",
]

outcome_fields = [
    "loan_identifier",
    "default_24m",
    "eligible_24m",
    "censored_24m",
    "observed_months_24m",
]


origination_model_source = pd.read_parquet(
    origination_path,
    columns=origination_fields,
)

outcome_model_source = pd.read_parquet(
    outcomes_path,
    columns=outcome_fields,
)


modeling_population = (
    origination_model_source.merge(
        outcome_model_source,
        on="loan_identifier",
        how="left",
        validate="one_to_one",
    )
)


# Assign each vintage its methodological role.
modeling_population["population_role"] = (
    modeling_population["vintage"].map(
        {
            2006: "historical_comparison",
            2015: "model_development",
            2016: "model_development",
            2017: "later_vintage_test",
        }
    )
)


# Only eligible 2015–2017 loans enter supervised modeling.
modeling_population["model_eligible"] = (
    modeling_population["eligible_24m"]
    & modeling_population["vintage"].isin(
        [2015, 2016, 2017]
    )
)


merge_validation = pd.Series(
    {
        "modeling_population_rows": len(
            modeling_population
        ),
        "unique_loan_ids": (
            modeling_population[
                "loan_identifier"
            ].nunique()
        ),
        "duplicate_loan_ids": int(
            modeling_population[
                "loan_identifier"
            ].duplicated().sum()
        ),
        "missing_outcomes": int(
            modeling_population[
                "eligible_24m"
            ].isna().sum()
        ),
        "missing_population_roles": int(
            modeling_population[
                "population_role"
            ].isna().sum()
        ),
        "model_eligible_loans": int(
            modeling_population[
                "model_eligible"
            ].sum()
        ),
    },
    name="result",
)

population_role_summary = (
    modeling_population
    .groupby(
        [
            "vintage",
            "population_role",
        ],
        dropna=False,
    )
    .agg(
        total_loans=(
            "loan_identifier",
            "size",
        ),
        eligible_outcomes=(
            "eligible_24m",
            "sum",
        ),
        censored_outcomes=(
            "censored_24m",
            "sum",
        ),
        defaults_24m=(
            "default_24m",
            lambda values: values.eq(1).sum(),
        ),
        model_eligible_loans=(
            "model_eligible",
            "sum",
        ),
    )
    .reset_index()
)

display(merge_validation)
display(population_role_summary)

modeling_population_rows    200000
unique_loan_ids             200000
duplicate_loan_ids               0
missing_outcomes                 0
missing_population_roles         0
model_eligible_loans        124451
Name: result, dtype: int64

,vintage,population_role,total_loans,eligible_outcomes,censored_outcomes,defaults_24m,model_eligible_loans
0,2006,historical_comparison,50000,37555,12445,967,0
1,2015,model_development,50000,39911,10089,253,39911
2,2016,model_development,50000,42476,7524,330,42476
3,2017,later_vintage_test,50000,42064,7936,397,42064


In [5]:
numeric_features = [
    "credit_score",
    "mi_percentage",
    "original_cltv",
    "original_dti",
    "original_upb",
    "original_ltv",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers",
]

categorical_features = [
    "first_time_homebuyer_indicator",
    "number_of_units",
    "occupancy_status",
    "channel",
    "prepayment_penalty_indicator",
    "amortization_type",
    "property_state",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
    "special_eligibility_program",
    "harp_indicator",
    "property_valuation_method",
    "interest_only_indicator",
]

predictor_features = (
    numeric_features
    + categorical_features
)


eligible_model_population = (
    modeling_population.loc[
        modeling_population[
            "model_eligible"
        ],
        predictor_features
        + [
            "vintage",
            "default_24m",
            "population_role",
        ],
    ]
    .copy()
)


feature_audit_rows = []

for feature in predictor_features:
    feature_audit_rows.append(
        {
            "feature": feature,
            "feature_type": (
                "numeric"
                if feature in numeric_features
                else "categorical"
            ),
            "dtype": str(
                eligible_model_population[
                    feature
                ].dtype
            ),
            "missing_count": int(
                eligible_model_population[
                    feature
                ].isna().sum()
            ),
            "missing_percent": round(
                eligible_model_population[
                    feature
                ].isna().mean()
                * 100,
                2,
            ),
            "unique_nonmissing_values": int(
                eligible_model_population[
                    feature
                ].nunique(
                    dropna=True
                )
            ),
            "constant_feature": bool(
                eligible_model_population[
                    feature
                ].nunique(
                    dropna=True
                )
                <= 1
            ),
        }
    )

feature_audit = (
    pd.DataFrame(feature_audit_rows)
    .sort_values(
        [
            "constant_feature",
            "missing_percent",
            "feature",
        ],
        ascending=[
            False,
            False,
            True,
        ],
    )
    .reset_index(drop=True)
)


numeric_feature_summary = (
    eligible_model_population[
        numeric_features
    ]
    .describe(
        percentiles=[
            0.01,
            0.25,
            0.50,
            0.75,
            0.99,
        ]
    )
    .T
    .reset_index()
    .rename(
        columns={
            "index": "feature"
        }
    )
)


display(feature_audit)
display(numeric_feature_summary)

,feature,feature_type,dtype,missing_count,missing_percent,unique_nonmissing_values,constant_feature
0,property_valuation_method,categorical,Int64,124081,99.70,1,True
1,amortization_type,categorical,string,0,0.00,1,True
2,interest_only_indicator,categorical,string,0,0.00,1,True
3,prepayment_penalty_indicator,categorical,string,0,0.00,1,True
4,special_eligibility_program,categorical,string,120122,96.52,2,False
5,original_dti,numeric,Int64,7347,5.90,50,False
6,channel,categorical,string,0,0.00,3,False
7,credit_score,numeric,Int64,5,0.00,337,False
8,first_time_homebuyer_indicator,categorical,string,0,0.00,2,False
9,harp_indicator,categorical,string,0,0.00,2,False


,feature,count,mean,std,min,1%,25%,50%,75%,99%,max
0,credit_score,124446.0,747.929512,47.23169,456.0,626.0,715.0,757.0,787.0,816.0,834.0
1,mi_percentage,124451.0,6.67204,11.57686,0.0,0.0,0.0,0.0,12.0,30.0,40.0
2,original_cltv,124450.0,74.772519,17.613737,5.0,24.0,66.0,80.0,87.0,98.0,854.0
3,original_dti,117104.0,34.32548,9.379494,1.0,12.0,28.0,35.0,42.0,50.0,50.0
4,original_upb,124451.0,225353.834039,117171.730305,10000.0,48000.0,135000.0,204000.0,300000.0,582000.0,1000000.0
5,original_ltv,124450.0,74.124018,17.420643,3.0,24.0,65.0,79.0,85.0,97.0,260.0
6,original_interest_rate,124451.0,3.96916,0.483505,2.25,2.75,3.625,4.0,4.25,5.125,6.125
7,original_loan_term,124451.0,317.964452,74.374089,96.0,156.0,300.0,360.0,360.0,360.0,366.0
8,number_of_borrowers,124451.0,1.493335,0.499958,1.0,1.0,1.0,1.0,2.0,2.0,2.0


## Feature Selection and Transformation Policy

Candidate predictors were evaluated for missingness, variation,
cardinality, stability, and availability at origination.

The following fields are excluded:

- `property_valuation_method`: 99.70% missing and only one observed level;
- `special_eligibility_program`: 96.52% missing and sparsely distributed;
- `amortization_type`: constant;
- `interest_only_indicator`: constant; and
- `prepayment_penalty_indicator`: constant.

Missing numeric values will not cause loan deletion. Median imputation
will later be fitted using only the 2015–2016 development population.
Missing-value indicators will preserve information about original
missingness.

The high LTV and CLTV values are retained as legitimate disclosed
records, including HARP loans. To reduce excessive model influence:

- LTV and CLTV are capped at 200 for the continuous model features;
- separate indicators identify ratios above 100; and
- the original disclosed values remain available for audit evidence.

Original UPB is log-transformed to reduce right skew.

No performance or post-origination information is used as a predictor.

In [7]:
import numpy as np
excluded_features = pd.DataFrame(
    [
        {
            "feature": "property_valuation_method",
            "reason": (
                "99.70% missing and only one "
                "observed level"
            ),
        },
        {
            "feature": "special_eligibility_program",
            "reason": (
                "96.52% missing and unstable "
                "coverage"
            ),
        },
        {
            "feature": "amortization_type",
            "reason": "Constant feature",
        },
        {
            "feature": "interest_only_indicator",
            "reason": "Constant feature",
        },
        {
            "feature": "prepayment_penalty_indicator",
            "reason": "Constant feature",
        },
    ]
)


feature_engineered_population = (
    modeling_population.copy()
)


# Missing-value indicators.
for feature in [
    "credit_score",
    "original_dti",
    "original_ltv",
    "original_cltv",
]:
    feature_engineered_population[
        f"{feature}_missing"
    ] = (
        feature_engineered_population[
            feature
        ].isna()
        .astype("int8")
    )


# Transform original UPB while preserving the source field.
feature_engineered_population[
    "original_upb_log"
] = np.log1p(
    feature_engineered_population[
        "original_upb"
    ]
)


# Preserve high-ratio information separately.
feature_engineered_population[
    "original_ltv_above_100"
] = (
    feature_engineered_population[
        "original_ltv"
    ]
    .gt(100)
    .fillna(False)
    .astype("int8")
)

feature_engineered_population[
    "original_cltv_above_100"
] = (
    feature_engineered_population[
        "original_cltv"
    ]
    .gt(100)
    .fillna(False)
    .astype("int8")
)


# Limit extreme continuous influence without deleting loans.
feature_engineered_population[
    "original_ltv_capped_200"
] = (
    feature_engineered_population[
        "original_ltv"
    ].clip(upper=200)
)

feature_engineered_population[
    "original_cltv_capped_200"
] = (
    feature_engineered_population[
        "original_cltv"
    ].clip(upper=200)
)


final_numeric_features = [
    "credit_score",
    "mi_percentage",
    "original_dti",
    "original_upb_log",
    "original_ltv_capped_200",
    "original_cltv_capped_200",
    "original_interest_rate",
    "original_loan_term",
    "number_of_borrowers",
    "credit_score_missing",
    "original_dti_missing",
    "original_ltv_missing",
    "original_cltv_missing",
    "original_ltv_above_100",
    "original_cltv_above_100",
]

final_categorical_features = [
    "first_time_homebuyer_indicator",
    "number_of_units",
    "occupancy_status",
    "channel",
    "property_state",
    "property_type",
    "loan_purpose",
    "super_conforming_flag",
    "harp_indicator",
]

final_predictor_features = (
    final_numeric_features
    + final_categorical_features
)


# Explicit leakage-control check.
prohibited_predictor_fields = {
    "loan_identifier",
    "vintage",
    "population_role",
    "default_24m",
    "eligible_24m",
    "censored_24m",
    "observed_months_24m",
    "model_eligible",
    "performance_row_count",
    "current_actual_upb",
    "current_loan_delinquency_status",
    "loan_age",
    "modification_flag",
    "zero_balance_code",
    "actual_loss",
}

leakage_fields_found = sorted(
    set(final_predictor_features)
    & prohibited_predictor_fields
)


transformation_summary = pd.Series(
    {
        "final_predictor_count": len(
            final_predictor_features
        ),
        "numeric_predictors": len(
            final_numeric_features
        ),
        "categorical_predictors": len(
            final_categorical_features
        ),
        "excluded_features": len(
            excluded_features
        ),
        "ltv_above_100_loans": int(
            feature_engineered_population[
                "original_ltv_above_100"
            ].sum()
        ),
        "cltv_above_100_loans": int(
            feature_engineered_population[
                "original_cltv_above_100"
            ].sum()
        ),
        "ltv_values_capped_above_200": int(
            feature_engineered_population[
                "original_ltv"
            ].gt(200).sum()
        ),
        "cltv_values_capped_above_200": int(
            feature_engineered_population[
                "original_cltv"
            ].gt(200).sum()
        ),
        "prohibited_predictors_found": len(
            leakage_fields_found
        ),
        "leakage_control_passed": (
            len(leakage_fields_found) == 0
        ),
    },
    name="result",
)

display(excluded_features)
display(transformation_summary)

,feature,reason
0,property_valuation_method,99.70% missing and only one observed level
1,special_eligibility_program,96.52% missing and unstable coverage
2,amortization_type,Constant feature
3,interest_only_indicator,Constant feature
4,prepayment_penalty_indicator,Constant feature


final_predictor_count             24
numeric_predictors                15
categorical_predictors             9
excluded_features                  5
ltv_above_100_loans              808
cltv_above_100_loans            1230
ltv_values_capped_above_200        5
cltv_values_capped_above_200       9
prohibited_predictors_found        0
leakage_control_passed          True
Name: result, dtype: object

In [8]:
traceability_fields = [
    "loan_identifier",
    "vintage",
    "population_role",
    "eligible_24m",
    "censored_24m",
    "default_24m",
]

modeling_dataset = (
    feature_engineered_population[
        traceability_fields
        + final_predictor_features
    ]
    .copy()
)


# Convert numeric predictors to standard floating-point
# values for later preprocessing.
for feature in final_numeric_features:
    modeling_dataset[feature] = (
        pd.to_numeric(
            modeling_dataset[feature],
            errors="coerce",
        )
        .astype("float64")
    )


# Keep categorical predictors consistently represented.
for feature in final_categorical_features:
    modeling_dataset[feature] = (
        modeling_dataset[feature]
        .astype("string")
    )


development_population = (
    modeling_dataset.loc[
        modeling_dataset[
            "vintage"
        ].isin([2015, 2016])
        & modeling_dataset[
            "eligible_24m"
        ]
    ]
    .copy()
)

test_population = (
    modeling_dataset.loc[
        modeling_dataset[
            "vintage"
        ].eq(2017)
        & modeling_dataset[
            "eligible_24m"
        ]
    ]
    .copy()
)

historical_comparison_population = (
    modeling_dataset.loc[
        modeling_dataset[
            "vintage"
        ].eq(2006)
        & modeling_dataset[
            "eligible_24m"
        ]
    ]
    .copy()
)


X_development = development_population[
    final_predictor_features
].copy()

y_development = (
    development_population[
        "default_24m"
    ]
    .astype("int8")
)

X_test = test_population[
    final_predictor_features
].copy()

y_test = (
    test_population[
        "default_24m"
    ]
    .astype("int8")
)

X_historical = (
    historical_comparison_population[
        final_predictor_features
    ]
    .copy()
)

y_historical = (
    historical_comparison_population[
        "default_24m"
    ]
    .astype("int8")
)


development_ids = set(
    development_population[
        "loan_identifier"
    ]
)

test_ids = set(
    test_population[
        "loan_identifier"
    ]
)

historical_ids = set(
    historical_comparison_population[
        "loan_identifier"
    ]
)


split_validation = pd.Series(
    {
        "development_rows": len(
            development_population
        ),
        "development_defaults": int(
            y_development.sum()
        ),
        "development_default_rate_percent": round(
            y_development.mean() * 100,
            3,
        ),
        "test_rows": len(
            test_population
        ),
        "test_defaults": int(
            y_test.sum()
        ),
        "test_default_rate_percent": round(
            y_test.mean() * 100,
            3,
        ),
        "historical_rows": len(
            historical_comparison_population
        ),
        "historical_defaults": int(
            y_historical.sum()
        ),
        "historical_default_rate_percent": round(
            y_historical.mean() * 100,
            3,
        ),
        "development_test_id_overlap": len(
            development_ids & test_ids
        ),
        "development_historical_id_overlap": len(
            development_ids & historical_ids
        ),
        "test_historical_id_overlap": len(
            test_ids & historical_ids
        ),
        "missing_development_targets": int(
            y_development.isna().sum()
        ),
        "missing_test_targets": int(
            y_test.isna().sum()
        ),
        "censored_loans_in_development": int(
            development_population[
                "censored_24m"
            ].sum()
        ),
        "censored_loans_in_test": int(
            test_population[
                "censored_24m"
            ].sum()
        ),
    },
    name="result",
)

display(split_validation)

development_rows                     82387.000
development_defaults                   583.000
development_default_rate_percent         0.708
test_rows                            42064.000
test_defaults                          397.000
test_default_rate_percent                0.944
historical_rows                      37555.000
historical_defaults                    967.000
historical_default_rate_percent          2.575
development_test_id_overlap              0.000
development_historical_id_overlap        0.000
test_historical_id_overlap               0.000
missing_development_targets              0.000
missing_test_targets                     0.000
censored_loans_in_development            0.000
censored_loans_in_test                   0.000
Name: result, dtype: float64

In [9]:
categorical_stability_rows = []

for feature in final_categorical_features:
    development_levels = set(
        X_development[
            feature
        ].dropna().unique()
    )

    test_levels = set(
        X_test[
            feature
        ].dropna().unique()
    )

    unseen_test_levels = sorted(
        test_levels - development_levels
    )

    unseen_test_rows = int(
        X_test[feature].isin(
            unseen_test_levels
        ).sum()
    )

    categorical_stability_rows.append(
        {
            "feature": feature,
            "development_levels": len(
                development_levels
            ),
            "test_levels": len(
                test_levels
            ),
            "unseen_test_level_count": len(
                unseen_test_levels
            ),
            "unseen_test_levels": (
                ", ".join(
                    map(str, unseen_test_levels)
                )
                if unseen_test_levels
                else ""
            ),
            "test_rows_with_unseen_level": (
                unseen_test_rows
            ),
        }
    )

categorical_stability = pd.DataFrame(
    categorical_stability_rows
)


numeric_missingness_rows = []

for feature in final_numeric_features:
    numeric_missingness_rows.append(
        {
            "feature": feature,
            "development_missing": int(
                X_development[
                    feature
                ].isna().sum()
            ),
            "development_missing_percent": round(
                X_development[
                    feature
                ].isna().mean()
                * 100,
                3,
            ),
            "test_missing": int(
                X_test[
                    feature
                ].isna().sum()
            ),
            "test_missing_percent": round(
                X_test[
                    feature
                ].isna().mean()
                * 100,
                3,
            ),
        }
    )

numeric_missingness = pd.DataFrame(
    numeric_missingness_rows
)


stability_summary = pd.Series(
    {
        "categorical_features_tested": len(
            categorical_stability
        ),
        "features_with_unseen_test_levels": int(
            categorical_stability[
                "unseen_test_level_count"
            ].gt(0).sum()
        ),
        "test_rows_with_any_unseen_category": int(
            pd.concat(
                [
                    X_test[feature].isin(
                        set(
                            X_test[
                                feature
                            ].dropna().unique()
                        )
                        - set(
                            X_development[
                                feature
                            ].dropna().unique()
                        )
                    )
                    for feature
                    in final_categorical_features
                ],
                axis=1,
            )
            .any(axis=1)
            .sum()
        ),
        "numeric_features_with_development_missingness": int(
            numeric_missingness[
                "development_missing"
            ].gt(0).sum()
        ),
        "numeric_features_with_test_missingness": int(
            numeric_missingness[
                "test_missing"
            ].gt(0).sum()
        ),
    },
    name="result",
)

display(categorical_stability)
display(numeric_missingness)
display(stability_summary)

,feature,development_levels,test_levels,unseen_test_level_count,unseen_test_levels,test_rows_with_unseen_level
0,first_time_homebuyer_indicator,2,2,0,,0
1,number_of_units,4,4,0,,0
2,occupancy_status,3,3,0,,0
3,channel,3,3,0,,0
4,property_state,54,53,0,,0
5,property_type,5,5,0,,0
6,loan_purpose,3,3,0,,0
7,super_conforming_flag,2,2,0,,0
8,harp_indicator,2,2,0,,0


,feature,development_missing,development_missing_percent,test_missing,test_missing_percent
0,credit_score,0,0.000,5,0.012
1,mi_percentage,0,0.000,0,0.000
2,original_dti,5764,6.996,1583,3.763
3,original_upb_log,0,0.000,0,0.000
4,original_ltv_capped_200,1,0.001,0,0.000
5,original_cltv_capped_200,1,0.001,0,0.000
6,original_interest_rate,0,0.000,0,0.000
7,original_loan_term,0,0.000,0,0.000
8,number_of_borrowers,0,0.000,0,0.000
9,credit_score_missing,0,0.000,0,0.000


categorical_features_tested                      9
features_with_unseen_test_levels                 0
test_rows_with_any_unseen_category               0
numeric_features_with_development_missingness    3
numeric_features_with_test_missingness           2
Name: result, dtype: int64

In [10]:
feature_manifest_rows = []

numeric_transformation_notes = {
    "credit_score": (
        "Development-median imputation; "
        "missingness indicator retained"
    ),
    "mi_percentage": "Retained as disclosed",
    "original_dti": (
        "Development-median imputation; "
        "missingness indicator retained"
    ),
    "original_upb_log": (
        "Natural log of Original UPB plus one"
    ),
    "original_ltv_capped_200": (
        "Original LTV capped at 200"
    ),
    "original_cltv_capped_200": (
        "Original CLTV capped at 200"
    ),
    "original_interest_rate": (
        "Retained as disclosed"
    ),
    "original_loan_term": (
        "Retained as disclosed"
    ),
    "number_of_borrowers": (
        "Retained as disclosed"
    ),
    "credit_score_missing": (
        "Binary source-missingness indicator"
    ),
    "original_dti_missing": (
        "Binary source-missingness indicator"
    ),
    "original_ltv_missing": (
        "Binary source-missingness indicator"
    ),
    "original_cltv_missing": (
        "Binary source-missingness indicator"
    ),
    "original_ltv_above_100": (
        "Binary high-LTV indicator"
    ),
    "original_cltv_above_100": (
        "Binary high-CLTV indicator"
    ),
}

for feature in final_numeric_features:
    feature_manifest_rows.append(
        {
            "feature": feature,
            "feature_type": "numeric",
            "source_timing": "Origination",
            "transformation": (
                numeric_transformation_notes[
                    feature
                ]
            ),
            "model_use": "Predictor",
        }
    )

for feature in final_categorical_features:
    feature_manifest_rows.append(
        {
            "feature": feature,
            "feature_type": "categorical",
            "source_timing": "Origination",
            "transformation": (
                "Development-fitted imputation "
                "and one-hot encoding"
            ),
            "model_use": "Predictor",
        }
    )

feature_manifest = pd.DataFrame(
    feature_manifest_rows
)


modeling_dataset_path = (
    processed_data_directory
    / "modeling_dataset.parquet"
)

feature_manifest_path = (
    processed_data_directory
    / "feature_manifest.csv"
)


modeling_dataset.to_parquet(
    modeling_dataset_path,
    index=False,
)

feature_manifest.to_csv(
    feature_manifest_path,
    index=False,
)


reloaded_modeling_dataset = (
    pd.read_parquet(
        modeling_dataset_path
    )
)

feature_export_validation = pd.Series(
    {
        "modeling_file_exists": (
            modeling_dataset_path.exists()
        ),
        "manifest_file_exists": (
            feature_manifest_path.exists()
        ),
        "exported_rows": len(
            reloaded_modeling_dataset
        ),
        "exported_columns": len(
            reloaded_modeling_dataset.columns
        ),
        "unique_exported_loan_ids": (
            reloaded_modeling_dataset[
                "loan_identifier"
            ].nunique()
        ),
        "duplicate_exported_loan_ids": int(
            reloaded_modeling_dataset[
                "loan_identifier"
            ].duplicated().sum()
        ),
        "eligible_outcomes": int(
            reloaded_modeling_dataset[
                "eligible_24m"
            ].sum()
        ),
        "censored_outcomes": int(
            reloaded_modeling_dataset[
                "censored_24m"
            ].sum()
        ),
        "manifest_predictors": len(
            feature_manifest
        ),
        "prohibited_predictors": len(
            set(final_predictor_features)
            & prohibited_predictor_fields
        ),
        "parquet_file_size_mb": round(
            modeling_dataset_path.stat().st_size
            / (1024 ** 2),
            2,
        ),
    },
    name="result",
)

display(feature_manifest)
display(feature_export_validation)

,feature,feature_type,source_timing,transformation,model_use
0,credit_score,numeric,Origination,Development-median imputation; missingness ind...,Predictor
1,mi_percentage,numeric,Origination,Retained as disclosed,Predictor
2,original_dti,numeric,Origination,Development-median imputation; missingness ind...,Predictor
3,original_upb_log,numeric,Origination,Natural log of Original UPB plus one,Predictor
4,original_ltv_capped_200,numeric,Origination,Original LTV capped at 200,Predictor
5,original_cltv_capped_200,numeric,Origination,Original CLTV capped at 200,Predictor
6,original_interest_rate,numeric,Origination,Retained as disclosed,Predictor
7,original_loan_term,numeric,Origination,Retained as disclosed,Predictor
8,number_of_borrowers,numeric,Origination,Retained as disclosed,Predictor
9,credit_score_missing,numeric,Origination,Binary source-missingness indicator,Predictor


modeling_file_exists             True
manifest_file_exists             True
exported_rows                  200000
exported_columns                   30
unique_exported_loan_ids       200000
duplicate_exported_loan_ids         0
eligible_outcomes              162006
censored_outcomes               37994
manifest_predictors                24
prohibited_predictors               0
parquet_file_size_mb             3.07
Name: result, dtype: object

## Feature Engineering and Leakage-Control Conclusion

A feature-engineered population was created for all 200,000 loans.

The exported dataset contains:

- six traceability and outcome-control fields;
- 15 numeric predictors;
- nine categorical predictors;
- 162,006 eligible outcomes;
- 37,994 censored outcomes; and
- one record per loan.

## Feature Decisions

Five candidate fields were excluded:

- `property_valuation_method` because it was 99.70% missing and had only
  one observed level;
- `special_eligibility_program` because it was 96.52% missing;
- `amortization_type` because it was constant;
- `interest_only_indicator` because it was constant; and
- `prepayment_penalty_indicator` because it was constant.

Missing numeric values were retained for development-fitted median
imputation. Missing-value indicators were created for credit score,
DTI, LTV, and CLTV.

Original UPB was log-transformed. LTV and CLTV were capped at 200 for
their continuous model features, while separate indicators preserved
whether their original values exceeded 100. No high-ratio loans were
deleted.

## Time-Based Population Design

| Population | Vintages | Eligible loans | Defaults | Default rate |
|---|---|---:|---:|---:|
| Development | 2015–2016 | 82,387 | 583 | 0.708% |
| Later-vintage test | 2017 | 42,064 | 397 | 0.944% |
| Historical comparison | 2006 | 37,555 | 967 | 2.575% |

No loan identifiers overlapped between the development, test, and
historical populations.

## Leakage Controls

All model predictors originate from information available at or before
loan origination.

The following were prohibited from model predictors:

- loan identifiers and population-assignment fields;
- outcome, eligibility, and censoring fields;
- monthly performance information;
- delinquency and servicing information;
- modification and zero-balance information; and
- actual-loss or other post-origination outcomes.

No prohibited predictors were identified.

Categorical stability testing found no 2017 categories that were absent
from the 2015–2016 development population.

All imputation, scaling, encoding, and model estimation will be fitted
using only the development population. The 2017 population will remain
untouched until model evaluation.

## Output Files

The validated outputs are:

- `data/processed/modeling_dataset.parquet`
- `data/processed/feature_manifest.csv`

The feature-engineered population is approved for baseline
probability-of-default modeling.